# routes/browser

> Route handlers for file browser navigation and file selection

In [ ]:
#| default_exp routes.browser

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Any, Tuple, Callable
from pathlib import Path

from fasthtml.common import APIRouter

from cjm_fasthtml_file_browser.core.config import FileBrowserConfig
from cjm_fasthtml_file_browser.providers.local import LocalFileSystemProvider
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile
from cjm_transcription_source_select.utils import detect_file_type
from cjm_transcription_source_select.routes.core import (
    get_step_state, update_step_state, get_session_id_from_sess
)
from cjm_transcription_source_select.components.file_browser_panel import (
    get_browser_state, sync_browser_selection, render_browser_panel
)

## Navigate Handler

Updates the current directory and re-renders the file browser.

In [ ]:
#| export
def _handle_navigate(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # Directory path to navigate to
):  # Rendered browser panel
    """Navigate to a directory and re-render the browser panel."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    browser_state = get_browser_state(step_state, home_path)

    # Validate path
    valid, _ = provider.is_valid_path(path)
    if valid and provider.is_directory(path):
        browser_state.current_path = provider.normalize_path(path)
    else:
        browser_state.current_path = step_state.get("current_directory", home_path)

    # Sync selection with selected_files
    selected_files = step_state.get("selected_files", [])
    sync_browser_selection(browser_state, selected_files)

    # Save state
    update_step_state(
        state_store, workflow_id, session_id,
        current_directory=browser_state.current_path,
        browser_state=browser_state.to_dict(),
    )

    return render_browser_panel(
        browser_state=browser_state,
        config=config,
        provider=provider,
        navigate_url=urls.navigate,
        select_url=urls.select,
        home_path=home_path,
    )

## Select Handler

Toggles a file in/out of the selected files list and re-renders the browser.

In [ ]:
#| export
def _handle_select(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    home_path: str,  # Home directory path
    urls: SourceSelectUrls,  # URL bundle
    sess,  # FastHTML session
    path: str,  # File path to toggle
):  # Rendered browser panel
    """Toggle a file in/out of the selected files list."""
    session_id = get_session_id_from_sess(sess)
    step_state = get_step_state(state_store, workflow_id, session_id)
    browser_state = get_browser_state(step_state, home_path)
    selected_files = step_state.get("selected_files", [])

    # Toggle: remove if present, add if not
    existing_paths = {f["path"] for f in selected_files}
    if path in existing_paths:
        selected_files = [f for f in selected_files if f["path"] != path]
    else:
        file_path = Path(path)
        file_type = detect_file_type(path)
        if file_type and file_path.exists():
            selected_files.append(SelectedFile(
                path=path,
                filename=file_path.name,
                file_type=file_type,
                size_bytes=file_path.stat().st_size,
                duration=None,
                format=file_path.suffix.lstrip("."),
            ))

    # Sync browser selection with updated list
    sync_browser_selection(browser_state, selected_files)

    # Clear verified state when selection changes
    update_step_state(
        state_store, workflow_id, session_id,
        selected_files=selected_files,
        verified=False,
        committed_audio_paths=[],
        browser_state=browser_state.to_dict(),
    )

    return render_browser_panel(
        browser_state=browser_state,
        config=config,
        provider=provider,
        navigate_url=urls.navigate,
        select_url=urls.select,
        home_path=home_path,
    )

## Router Initialization

In [ ]:
#| export
def init_browser_router(
    state_store: SQLiteWorkflowStateStore,  # Workflow state store
    provider: LocalFileSystemProvider,  # File system provider
    config: FileBrowserConfig,  # Browser configuration
    workflow_id: str,  # Workflow identifier
    urls: SourceSelectUrls,  # Mutable URL bundle (populated after router init)
    home_path: str = "",  # Home directory for nav buttons
    prefix: str = "/browser",  # Route prefix
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize the file browser router with navigate and select handlers."""
    router = APIRouter(prefix=prefix)
    _home = home_path or provider.get_home_path()

    @router
    def navigate(sess, path: str = ""):
        """Navigate to a directory."""
        target = path or _home
        return _handle_navigate(
            state_store, provider, config, workflow_id, _home, urls,
            sess, target,
        )

    @router
    def select(sess, path: str = ""):
        """Toggle a file in/out of the selection."""
        if not path:
            return navigate(sess)
        return _handle_select(
            state_store, provider, config, workflow_id, _home, urls,
            sess, path,
        )

    routes = {
        "navigate": navigate,
        "select": select,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()